# PI III — Motor de Busca (Baixada Santista)

### 1. Configuração e download dos artigos da Wikipédia

Esta primeira parte carrega a biblioteca `httr2` e define a função `baixar_wiki`, que recebe o
título de um artigo e consulta a API da Wikipédia em português (`action=query`, `prop=extracts`,
`explaintext=1`) para obter o texto puro (sem marcação wiki) do artigo. Se o artigo não existir,
a função interrompe a execução com `stop()`.

Em seguida, um vetor nomeado `municipios` associa um apelido curto (`santos`, `praia_grande`,
`sao_vicente`) ao título exato de cada artigo na Wikipédia, e `sapply` aplica `baixar_wiki` a
cada um deles, produzindo o vetor `docs` com o texto completo de cada município. Por fim, o
código imprime o tamanho (em caracteres) de cada artigo baixado — uma primeira noção do volume
de texto que será processado.

In [ ]:
library(httr2)

baixar_wiki <- function(titulo) {

  resposta <- request(
    "https://pt.wikipedia.org/w/api.php"
  ) |>
    req_url_query(
      action = "query",
      prop = "extracts",
      explaintext = 1,
      format = "json",
      redirects = 1,
      titles = titulo
    ) |>
    req_perform() |>
    resp_body_json()

  pagina <- resposta$query$pages[[1]]

  if (is.null(pagina$extract)) {
    stop(
      paste(
        "Não foi possível encontrar o artigo:",
        titulo
      )
    )
  }

  return(pagina$extract)
}

municipios <- c(
  santos = "Bolsa do Café",
  praia_grande = "Fortaleza de Itaipu",
  sao_vicente = "São Vicente (São Paulo)"
)

docs <- sapply(
  municipios,
  baixar_wiki
)

cat("============================================\n")
cat("       TAMANHO DOS ARTIGOS\n")
cat("============================================\n\n")
for (i in seq_along(docs)) {
  cat(
    "-",
    names(docs)[i],
    ":",
    nchar(docs[[i]]),
    "caracteres\n"
  )
}

       TAMANHO DOS ARTIGOS

- santos : 5510 caracteres
- praia_grande : 3456 caracteres
- sao_vicente : 13227 caracteres


### 2. Segmentação de frases

Esta seção define separa_frases, que quebra um texto em frases individuais usando uma
expressão regular com *lookbehind* (`(?<=[.!?])\s+`): ela divide o texto sempre que encontra
um ou mais espaços logo após um ponto final, exclamação ou interrogação, preservando a
pontuação em cada frase. Depois disso, trimws remove espaços nas pontas e frases vazias
(resultantes de quebras de linha nos artigos) são descartadas.

A função é aplicada aos três artigos, gerando santos_frases, praia_grande_frases e
sao_vicente_frases. O código imprime quantas frases cada município tem e mostra as três
primeiras de cada um, como amostra. Por fim, todas as frases são combinadas em um único vetor
frases, e origem_frases guarda, para cada posição desse vetor, de qual município ela veio —
isso será usado mais adiante para saber a origem de cada frase nas comparações de similaridade.

In [ ]:
separa_frases <- function(texto) {
  frases <- unlist(
    strsplit(
      texto,
      "(?<=[.!?])\\s+",
      perl = TRUE
    )
  )
  frases <- trimws(frases)
  frases <- frases[
    frases != ""
  ]
  return(frases)
}

santos_frases <- separa_frases(docs[["santos"]])
praia_grande_frases <- separa_frases(docs[["praia_grande"]])
sao_vicente_frases <- separa_frases(docs[["sao_vicente"]])

cat("          FRASES DOS CORPUS\n")
cat("- Santos:", length(santos_frases), "frases\n")
cat("- Praia Grande:", length(praia_grande_frases), "frases\n")
cat("- São Vicente:", length(sao_vicente_frases), "frases\n")

cat("       PRIMEIRAS FRASES\n")
cat("\n--- SANTOS ---\n")
print(head(santos_frases, 3))
cat("\n--- PRAIA GRANDE ---\n")
print(head(praia_grande_frases, 3))
cat("\n--- SÃO VICENTE ---\n")
print(head(sao_vicente_frases, 3))

frases <- c(santos_frases, praia_grande_frases, sao_vicente_frases)
origem_frases <- c(
  rep("santos", length(santos_frases)),
  rep("praia_grande", length(praia_grande_frases)),
  rep("sao_vicente", length(sao_vicente_frases))
)

          FRASES DOS CORPUS
- Santos: 25 frases
- Praia Grande: 33 frases
- São Vicente: 78 frases
       PRIMEIRAS FRASES

--- SANTOS ---
[1] "Bolsa de Café, ou o Palácio da Bolsa Oficial de Café, foi um centro de negociação de café e atualmente é um museu localizado na rua XV de Novembro, no centro histórico do município de Santos, estado de São Paulo, Brasil."
[2] "Após um restauro realizado em 1998, o palácio foi reinaugurado como o Museu do Café."                                                                                                                                        
[3] "== Antecedentes ==\nA Bolsa Oficial do Café, em Santos, foi criada pela Lei Estadual no 1416, de 14 de julho de 1914, para atender ao grande movimento comercial do café na cidade de Santos."                               

--- PRAIA GRANDE ---
[1] "Forte Duque de Caxias de Itaipu, conhecido por Fortaleza de Itaipu, localiza-se na Ponta de Itaipu, em Praia Grande, dominando a barra de São Vicente, no

### 3. Tokenização, limpeza e stemming

Esta seção define a lista de *stopwords* em português (palavras muito frequentes e pouco
informativas, como "de", "o", "a", "que") e a função `tokenizar_limpar`, que:

1. converte o texto para minúsculas;
2. remove números e pontuação, substituindo-os por espaço;
3. quebra o texto em tokens (palavras) separando por espaços em branco;
4. descarta tokens vazios, tokens que são stopwords e tokens com 2 caracteres ou menos;
5. aplica stemming com `SnowballC::wordStem(..., "portuguese")`, reduzindo cada palavra ao
   seu radical comum — por exemplo, "documentos" e "documento" colapsam
   no mesmo termo, assim como "café" e "cafeteria". Isso reduz o tamanho do vocabulário e deixa
   o índice mais "limpo", ao custo de o radical nem sempre ser uma palavra real (trade-off entre
   recall e precisão, como alerta o slide).

A função é aplicada a cada um dos três documentos com `lapply`, gerando a lista `tokens`.
O código então imprime quantos tokens sobraram em cada documento depois de toda essa limpeza.

In [ ]:
install.packages("SnowballC")  # só na primeira vez na sessão
library(SnowballC)

stopwords <- c(
  "a", "à", "ao", "aos", "as",
  "às", "até",
  "com", "como",
  "da", "das", "de", "do", "dos",
  "e", "é", "em", "entre",
  "era", "eram",
  "essa", "essas", "esse", "esses",
  "esta", "estas", "este", "estes",
  "foi", "foram",
  "há",
  "isso", "isto",
  "já",
  "mas",
  "mais",
  "me", "mesmo",
  "na", "nas", "nem", "no", "nos",
  "não",
  "o", "os",
  "ou",
  "para", "pela", "pelas", "pelo", "pelos",
  "por",
  "qual", "quando", "que", "quem",
  "se", "sem", "ser",
  "seu", "seus",
  "sua", "suas",
  "também",
  "tem", "têm",
  "um", "uma", "umas", "uns",
  "vai", "vão"
)

tokenizar_limpar <- function(texto) {
  texto <- tolower(texto)
  texto <- gsub("[0-9]+", " ", texto)
  texto <- gsub("[[:punct:]]+", " ", texto)

  tokens <- unlist(strsplit(texto, "\\s+"))
  tokens <- tokens[tokens != ""]
  tokens <- tokens[!tokens %in% stopwords]
  tokens <- tokens[nchar(tokens) > 2]

  # stemming (slide 11): reduz variações da mesma palavra a um radical comum
  tokens <- wordStem(tokens, language = "portuguese")

  return(tokens)
}

tokens <- lapply(docs, tokenizar_limpar)

cat("\n============================================\n")
cat("       APÓS A LIMPEZA (com stemming)\n")
cat("============================================\n\n")
for (i in seq_along(tokens)) {
  cat("-", names(tokens)[i], ":", length(tokens[[i]]), "tokens\n")
}

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)




       APÓS A LIMPEZA (com stemming)

- santos : 499 tokens
- praia_grande : 307 tokens
- sao_vicente : 1223 tokens


### 4. Vocabulário e frequência de termos

`vocab` reúne todos os tokens de todos os documentos (`unlist(tokens)`), remove duplicatas com
`unique` e ordena alfabeticamente — esse é o vocabulário do corpus, ou seja, o conjunto de todos
os radicais distintos que aparecem em pelo menos um documento. O tamanho desse vetor
(`length(vocab)`) mostra quantos termos diferentes existem no total.

Em seguida, `table(unlist(tokens))` conta quantas vezes cada termo aparece somando as três
coleções, e `sort(..., decreasing = TRUE)` mais `head(..., 10)` extraem os 10 termos mais
frequentes do corpus inteiro — uma visão geral de quais palavras dominam o vocabulário (o que,
como o slide 30 comenta, tende a coincidir com termos de menor IDF).

In [ ]:
vocab <- sort(unique(unlist(tokens)))

cat("\n============================================\n")
cat("             VOCABULÁRIO\n")
cat("============================================\n\n")
cat("Quantidade de termos diferentes:", length(vocab), "\n")

freq <- table(unlist(tokens))
top10 <- head(sort(freq, decreasing = TRUE), 10)

cat("\n============================================\n")
cat("       10 TERMOS MAIS FREQUENTES\n")
cat("============================================\n\n")
for (i in seq_along(top10)) {
  cat(i, "º -", names(top10)[i], ":", as.integer(top10[i]), "ocorrências\n")
}


             VOCABULÁRIO

Quantidade de termos diferentes: 1028 

       10 TERMOS MAIS FREQUENTES

1 º - sã : 59 ocorrências
2 º - vicent : 39 ocorrências
3 º - caf : 38 ocorrências
4 º - cidad : 23 ocorrências
5 º - brasil : 22 ocorrências
6 º - sant : 21 ocorrências
7 º - vil : 21 ocorrências
8 º - histór : 20 ocorrências
9 º - paul : 19 ocorrências
10 º - primeir : 17 ocorrências


### 5. Matriz termo-documento (TDM)

Aqui é construída a matriz termo-documento: para cada documento, `table(factor(tk, levels =
vocab))` conta quantas vezes cada termo do vocabulário aparece naquele documento (usar
`levels = vocab` garante que todo termo do vocabulário tenha uma linha, mesmo que a contagem
seja zero para algum documento). `sapply` aplica isso aos três documentos, montando uma matriz
onde cada coluna é um documento e cada linha é um termo; `rownames(tdm) <- vocab` nomeia as
linhas com os termos correspondentes.

O código então imprime as dimensões da matriz (número de termos × número de documentos) e uma
prévia das 10 primeiras linhas, para dar uma ideia visual de como os termos se distribuem entre
os três municípios.

In [ ]:
tdm <- sapply(
  tokens,
  function(tk) {
    as.integer(table(factor(tk, levels = vocab)))
  }
)
rownames(tdm) <- vocab

cat("\n============================================\n")
cat("       MATRIZ TERMO-DOCUMENTO\n")
cat("============================================\n\n")
cat("Número de termos:", nrow(tdm), "\n")
cat("Número de documentos:", ncol(tdm), "\n")
cat("Dimensão:", nrow(tdm), "x", ncol(tdm), "\n")
cat("\nPrimeiros 10 termos:\n\n")
print(tdm[1:min(10, nrow(tdm)), ])


       MATRIZ TERMO-DOCUMENTO

Número de termos: 1028 
Número de documentos: 3 
Dimensão: 1028 x 3 

Primeiros 10 termos:

        santos praia_grande sao_vicente
abandon      1            0           0
abert        1            1           0
abrig        0            0           2
abril        0            1           0
abund        0            0           1
acerv        2            0           0
acess        0            3           2
achou        0            0           1
acident      0            0           1
acim         0            0           1


### 6. Cálculo do TF-IDF

Esta seção calcula o TF-IDF (Term Frequency–Inverse Document Frequency) de cada termo em cada
documento, uma medida de quão importante uma palavra é para um documento dentro da coleção.

- `N` é o número total de documentos (3, nesse caso).
- `total_termos` soma, por documento, a quantidade total de tokens (`colSums(tdm)`).
- `tf` (frequência do termo) divide cada contagem da `tdm` pelo total de termos do respectivo
  documento (`sweep(tdm, 2, total_termos, "/")`), normalizando por tamanho do documento.
- `documentos_com_termo` conta em quantos documentos cada termo aparece pelo menos uma vez
  (`rowSums(tdm > 0)`).
- `idf` (frequência inversa nos documentos) é `log(N / documentos_com_termo)`: quanto mais raro
  o termo entre os documentos, maior o IDF.
- `tfidf` é o produto elemento a elemento de `tf` e `idf`.

Por fim, para cada documento o código filtra os termos com TF-IDF positivo, ordena do maior para
o menor e imprime os 10 termos mais característicos daquele documento — os que melhor o
distinguem dos demais.

In [ ]:
N <- ncol(tdm)
total_termos <- colSums(tdm)
tf <- sweep(tdm, 2, total_termos, "/")
documentos_com_termo <- rowSums(tdm > 0)
idf <- log(N / documentos_com_termo)
tfidf <- tf * idf

cat("\n============================================\n")
cat("                 TF-IDF\n")
cat("============================================\n\n")
cat("Número de termos:", nrow(tfidf), "\n")
cat("Número de documentos:", ncol(tfidf), "\n")

for (documento in colnames(tfidf)) {
  cat("\n--------------------------------------------\n")
  cat("Termos mais importantes para:", documento, "\n")
  cat("--------------------------------------------\n\n")
  valores <- tfidf[, documento]
  valores <- valores[valores > 0]
  top <- head(sort(valores, decreasing = TRUE), 10)
  for (i in seq_along(top)) {
    cat(i, "º -", names(top)[i], ":", round(top[i], 4), "\n")
  }
}


                 TF-IDF

Número de termos: 1028 
Número de documentos: 3 

--------------------------------------------
Termos mais importantes para: santos 
--------------------------------------------

1 º - caf : 0.0837 
2 º - bols : 0.0286 
3 º - mus : 0.0198 
4 º - paláci : 0.011 
5 º - cafet : 0.0088 
6 º - cri : 0.0088 
7 º - trabalh : 0.0088 
8 º - cont : 0.0066 
9 º - document : 0.0066 
10 º - obra : 0.0066 

--------------------------------------------
Termos mais importantes para: praia_grande 
--------------------------------------------

1 º - fort : 0.0358 
2 º - artilh : 0.0215 
3 º - canhõ : 0.0179 
4 º - barr : 0.0143 
5 º - bat : 0.0143 
6 º - itaipu : 0.0143 
7 º - canet : 0.0107 
8 º - fortific : 0.0107 
9 º - garr : 0.0107 
10 º - schneid : 0.0107 

--------------------------------------------
Termos mais importantes para: sao_vicente 
--------------------------------------------

1 º - vicent : 0.0123 
2 º - vil : 0.0063 
3 º - cidad : 0.006 
4 º - afons : 0.0054

### 7. Similaridade de cosseno entre documentos

`similaridade_cosseno(v1, v2)` calcula a similaridade de cosseno entre dois vetores — nesse caso,
as colunas de TF-IDF de dois documentos. Primeiro substitui eventuais `NaN` por 0 (podem surgir
se um documento não tiver nenhum termo do vocabulário). Depois calcula o produto escalar
(`sum(v1 * v2)`) e as normas de cada vetor (`sqrt(sum(v^2))`); se alguma norma for 0 (vetor
nulo), a função retorna 0 para evitar divisão por zero. Caso contrário, retorna o produto escalar
dividido pelo produto das normas — o cosseno do ângulo entre os dois vetores, que varia de 0
(nenhuma relação) a 1 (idênticos em direção).

Em seguida, o código monta uma matriz quadrada `similaridade` (documentos × documentos) e
preenche cada célula `[i, j]` com a similaridade de cosseno entre as colunas `i` e `j` da
`tfidf`, comparando todos os pares de documentos entre si — inclusive cada documento consigo
mesmo (que dá sempre 1). O resultado final é impresso arredondado a 4 casas decimais.

In [ ]:
similaridade_cosseno <- function(v1, v2) {
  v1[is.nan(v1)] <- 0
  v2[is.nan(v2)] <- 0
  produto_escalar <- sum(v1 * v2)
  norma_v1 <- sqrt(sum(v1^2))
  norma_v2 <- sqrt(sum(v2^2))
  if (is.nan(norma_v1) || is.nan(norma_v2) || norma_v1 == 0 || norma_v2 == 0) {
    return(0)
  }
  return(produto_escalar / (norma_v1 * norma_v2))
}

similaridade <- matrix(0, nrow = ncol(tfidf), ncol = ncol(tfidf))
rownames(similaridade) <- colnames(tfidf)
colnames(similaridade) <- colnames(tfidf)

for (i in 1:ncol(tfidf)) {
  for (j in 1:ncol(tfidf)) {
    similaridade[i, j] <- similaridade_cosseno(tfidf[, i], tfidf[, j])
  }
}

cat("\n============================================\n")
cat("       SIMILARIDADE DE COSSENO\n")
cat("============================================\n\n")
print(round(similaridade, 4))


       SIMILARIDADE DE COSSENO

             santos praia_grande sao_vicente
santos       1.0000       0.0022      0.0193
praia_grande 0.0022       1.0000      0.0324
sao_vicente  0.0193       0.0324      1.0000


### 8. TF-IDF e similaridade de cosseno por frase

Esta parte repete todo o processo das seções 4 a 7, mas agora tratando **cada frase** (e não
cada documento inteiro) como uma unidade: `tokens_frases` aplica `tokenizar_limpar` a cada
elemento do vetor `frases`; `vocab_frases` é o vocabulário resultante; `tdm_frases` é a matriz
termo-frase (colunas nomeadas `frase_1`, `frase_2`, ...); e `tf_frases`/`idf_frases`/
`tfidf_frases` seguem exatamente a mesma fórmula da seção 6, só que em nível de frase.

Por fim, `similaridade_frases` é preenchida com a similaridade de cosseno entre todos os pares
de frases (usando a mesma função `similaridade_cosseno` da seção 7). Como há dezenas de frases,
essa matriz é bem maior do que a de documentos — ela vai alimentar a próxima seção, que extrai
os pares de frases mais parecidas entre si.

In [ ]:
tokens_frases <- lapply(frases, tokenizar_limpar)
vocab_frases <- sort(unique(unlist(tokens_frases)))

tdm_frases <- sapply(
  tokens_frases,
  function(tk) as.integer(table(factor(tk, levels = vocab_frases)))
)
rownames(tdm_frases) <- vocab_frases
colnames(tdm_frases) <- paste0("frase_", seq_along(frases))

N_frases <- ncol(tdm_frases)
total_termos_frase <- colSums(tdm_frases)
tf_frases <- sweep(tdm_frases, 2, total_termos_frase, "/")
documentos_com_termo_frase <- rowSums(tdm_frases > 0)
idf_frases <- log(N_frases / documentos_com_termo_frase)
tfidf_frases <- tf_frases * idf_frases

similaridade_frases <- matrix(0, nrow = ncol(tfidf_frases), ncol = ncol(tfidf_frases))
rownames(similaridade_frases) <- colnames(tfidf_frases)
colnames(similaridade_frases) <- colnames(tfidf_frases)

for (i in 1:ncol(tfidf_frases)) {
  for (j in 1:ncol(tfidf_frases)) {
    similaridade_frases[i, j] <- similaridade_cosseno(tfidf_frases[, i], tfidf_frases[, j])
  }
}

### 9. Pares de frases mais semelhantes

O código monta um `data.frame` vazio `resultados` (com colunas `frase_1`, `frase_2`,
`origem_1`, `origem_2` e `similaridade`) e depois percorre todos os pares únicos de frases
`(i, j)` com `i < j` — o laço duplo com `j` começando em `i + 1` evita comparar uma frase
consigo mesma e evita contar o mesmo par duas vezes (frase A com B e depois B com A). Para cada
par, adiciona uma linha ao `data.frame` com o texto das duas frases, seus municípios de origem
(vindos de `origem_frases`) e a similaridade de cosseno correspondente (lida da matriz
`similaridade_frases` calculada na seção 8).

Depois de montar todos os pares, `resultados` é ordenado por similaridade decrescente, e o
código imprime os 10 pares mais semelhantes — útil para identificar, por exemplo, frases quase
duplicadas ("boilerplate" de referências e links externos) ou frases de municípios diferentes
que tratam de temas parecidos (como as duas versões da lenda de descobrimento de São Vicente).

In [ ]:
resultados <- data.frame(
  frase_1 = character(),
  frase_2 = character(),
  origem_1 = character(),
  origem_2 = character(),
  similaridade = numeric(),
  stringsAsFactors = FALSE
)

for (i in 1:(ncol(tfidf_frases) - 1)) {
  for (j in (i + 1):ncol(tfidf_frases)) {
    resultados <- rbind(
      resultados,
      data.frame(
        frase_1 = frases[i],
        frase_2 = frases[j],
        origem_1 = origem_frases[i],
        origem_2 = origem_frases[j],
        similaridade = similaridade_frases[i, j],
        stringsAsFactors = FALSE
      )
    )
  }
}

resultados <- resultados[order(resultados$similaridade, decreasing = TRUE), ]

cat("\n============================================\n")
cat("      10 FRASES MAIS SEMELHANTES\n")
cat("============================================\n\n")

top_pares <- head(resultados, 10)
for (i in seq_len(nrow(top_pares))) {
  cat("\n--------------------------------------------\n")
  cat("Comparação", i, "\n")
  cat("Origem 1:", top_pares$origem_1[i], "\n")
  cat("Frase 1:", top_pares$frase_1[i], "\n")
  cat("\nOrigem 2:", top_pares$origem_2[i], "\n")
  cat("Frase 2:", top_pares$frase_2[i], "\n")
  cat("\nSimilaridade de cosseno:", round(top_pares$similaridade[i], 4), "\n")
}


      10 FRASES MAIS SEMELHANTES


--------------------------------------------
Comparação 1 
Origem 1: praia_grande 
Frase 1: Fortificações no Brasil (Resumo Histórico). 

Origem 2: praia_grande 
Frase 2: Fortificações do Brasil. 

Similaridade de cosseno: 0.6299 

--------------------------------------------
Comparação 2 
Origem 1: sao_vicente 
Frase 1: A ilha onde a cidade se estabeleceu havia sido descoberta anteriormente, em 1502, durante a expedição de Gaspar de Lemos e Américo Vespúcio, que a batizaram em homenagem a São Vicente Mártir, tornando-a a primeira vila da América Portuguesa. 

Origem 2: sao_vicente 
Frase 2: A Ilha de Gohayó foi descoberta em 22 de janeiro de 1502, pela expedição portuguesa comandada por Gaspar de Lemos e Américo Vespúcio, e, por ser o dia do santo mártir Vicente de Saragoça, foi batizada como Ilha de São Vicente. 

Similaridade de cosseno: 0.5838 

--------------------------------------------
Comparação 3 
Origem 1: santos 
Frase 1: Bolsa de Café, o

### 10. Índice invertido (postings)

Diferente de consultar a `tdm` inteira a cada busca, esta seção constrói um **índice invertido**
de verdade, seguindo os slides 22 a 27: um dicionário `postings` em que cada termo aponta para a
lista de documentos em que aparece (a relação é literalmente invertida — de "documento → termos"
para "termo → documentos").

O laço percorre cada documento (`for (d in names(docs))`), tokeniza e limpa seu texto,
usa `unique()` para não repetir o mesmo documento na lista de um termo caso ele apareça mais de
uma vez, e para cada termo distinto anexa o nome do documento (`postings[[termo]] <- c(...)`).
Se o termo ainda não existe em `postings`, `postings[[termo]]` é `NULL` e o `c()` simplesmente
cria a lista com um elemento — não é necessário inicializar cada entrada manualmente.

Por fim, o código imprime quantos termos foram indexados e quais têm as listas de postagens mais
longas (ou seja, aparecem em mais documentos — normalmente os de menor IDF).

In [ ]:
postings <- list()

for (d in names(docs)) {                                  # 1) para cada documento...
  for (termo in unique(tokenizar_limpar(docs[[d]]))) {     # 2) cada termo distinto...
    postings[[termo]] <- c(postings[[termo]], d)           # 3) anexa o documento à lista do termo
  }
}

cat("\n============================================\n")
cat("       ÍNDICE INVERTIDO (POSTINGS)\n")
cat("============================================\n\n")
cat("Termos indexados:", length(postings), "\n\n")

cat("Termos com mais documentos associados:\n")
print(sort(lengths(postings), decreasing = TRUE)[1:5])


       ÍNDICE INVERTIDO (POSTINGS)

Termos indexados: 1028 

Termos com mais documentos associados:
oficial localiz  histór    sant   estad 
      3       3       3       3       3 


### 11. `busca_AND` e `busca_OR`

Com o índice invertido pronto, dá para responder consultas com múltiplos termos sem reler
nenhum documento — apenas cruzando as listas de postagens já calculadas (slides 28-29).

- `busca_AND(consulta)` tokeniza a consulta com a **mesma** `tokenizar_limpar` usada para
  indexar os documentos (incluindo o stemming da seção 3 — essa é a "regra de ouro" do slide 29:
  se a consulta não passar pelo mesmo pré-processamento, "Documentos" não encontraria
  "documentos"), descarta termos que não existem no índice, e usa `Reduce(intersect, ...)` para
  intersectar as listas de postagens de todos os termos, dois a dois — o resultado são os
  documentos que contêm **todos** os termos da consulta.
- `busca_OR(consulta)` faz o mesmo, mas com `Reduce(union, ...)` — o resultado são os documentos
  que contêm **pelo menos um** dos termos.

A função auxiliar `mostrar_resultado` só formata a saída. No final, o código compara `busca_AND`
e `busca_OR` para a consulta "café museu" (mostrando a diferença entre exigir todos os termos ou
só um deles) e também busca "porto" isoladamente, para comparar com o resultado da busca
booleana simples que veremos a seguir.

In [ ]:
busca_AND <- function(consulta) {
  termos <- tokenizar_limpar(consulta)
  termos <- termos[termos %in% names(postings)]
  if (length(termos) == 0) {
    return(character(0))
  }
  Reduce(intersect, postings[termos])
}

busca_OR <- function(consulta) {
  termos <- tokenizar_limpar(consulta)
  termos <- termos[termos %in% names(postings)]
  if (length(termos) == 0) {
    return(character(0))
  }
  Reduce(union, postings[termos])
}

mostrar_resultado <- function(titulo, docs_encontrados) {
  cat("\n--------------------------------------------\n")
  cat(titulo, "\n")
  cat("--------------------------------------------\n")
  if (length(docs_encontrados) == 0) {
    cat("Nenhum documento encontrado.\n")
  } else {
    for (d in docs_encontrados) cat("-", d, "\n")
  }
}

cat("\n============================================\n")
cat("       BUSCA_AND vs BUSCA_OR\n")
cat("============================================\n")

consulta_teste <- "café museu"

mostrar_resultado(
  paste0("busca_AND('", consulta_teste, "')  — precisa ter TODOS os termos"),
  busca_AND(consulta_teste)
)

mostrar_resultado(
  paste0("busca_OR('", consulta_teste, "')  — precisa ter PELO MENOS UM termo"),
  busca_OR(consulta_teste)
)

mostrar_resultado(
  "busca_AND('porto')",
  busca_AND("porto")
)


       BUSCA_AND vs BUSCA_OR

--------------------------------------------
busca_AND('café museu')  — precisa ter TODOS os termos 
--------------------------------------------
- santos 

--------------------------------------------
busca_OR('café museu')  — precisa ter PELO MENOS UM termo 
--------------------------------------------
- santos 

--------------------------------------------
busca_AND('porto') 
--------------------------------------------
- santos 
- praia_grande 
- sao_vicente 


### 12. Busca booleana original (para comparação)

Esta função é a versão mais simples de busca, mantida aqui como referência. Ela recebe um único
`termo` e a matriz termo-documento `tdm`; converte o termo para minúsculas (mas, ao contrário de
`tokenizar_limpar`, não remove pontuação/acentos nem aplica stopwords ou stemming); verifica se
o termo existe entre as linhas da `tdm` (`rownames(tdm)`) — se não existir, retorna um vetor de
caracteres vazio; se existir, retorna os nomes das colunas (documentos) em que a contagem daquele
termo é maior que zero.

Como essa função não faz stemming na consulta, para buscar um termo que foi indexado com radical
(como "porto") é preciso passá-lo já no formato do radical (`wordStem("porto", "portuguese")`),
o que evidencia justamente a vantagem de `busca_AND`/`busca_OR` da seção 11, que cuidam disso
automaticamente.

In [ ]:
busca_booleana <- function(termo, tdm) {
  termo <- tolower(termo)
  if (!termo %in% rownames(tdm)) {
    return(character(0))
  }
  colnames(tdm)[tdm[termo, ] > 0]
}

cat("\n============================================\n")
cat("       BUSCA BOOLEANA (versão original)\n")
cat("============================================\n\n")

termo_pesquisado <- wordStem("porto", language = "portuguese")  # precisa bater com o radical indexado

cat("Termo pesquisado (radical):", termo_pesquisado, "\n\n")

documentos_encontrados <- busca_booleana(termo_pesquisado, tdm)

if (length(documentos_encontrados) == 0) {
  cat("O termo não foi encontrado.\n")
} else {
  cat("O termo aparece nos documentos:\n")
  for (documento in documentos_encontrados) {
    cat("-", documento, "\n")
  }
}


       BUSCA BOOLEANA (versão original)

Termo pesquisado (radical): port 

O termo aparece nos documentos:
- santos 
- praia_grande 
- sao_vicente 


### 13. BM25 — tamanho dos documentos e IDF probabilístico

Esta seção prepara os dois ingredientes do BM25 que o TF-IDF não usa: o tamanho de cada
documento (para a normalização por `b`) e uma versão probabilística do IDF (para a
saturação por `k1`).

`dl` reaproveita `colSums(tdm)` — o mesmo cálculo que `total_termos` já fazia na seção 6,
só que agora com o nome `dl` (*document length*) usado nos slides. `avgdl` é a média
desses tamanhos: os artigos de Santos, Praia Grande e São Vicente têm tamanhos bem
diferentes entre si, então essa comparação vai pesar bastante no `K` de cada documento.

`idf_bm25` usa a fórmula do slide 27 — `log((N - df + 0.5) / (df + 0.5) + 1)` — em vez do
`log(N/df)` clássico da seção 6. Ela reaproveita `N` e `documentos_com_termo`, que já
estavam calculados. Diferente do IDF clássico, esta versão nunca fica negativa (por causa
do `+1`), mesmo para termos muito comuns como "sã" ou "vicent".

In [ ]:
dl <- colSums(tdm)      # |d|: tamanho de cada documento (em tokens já processados)
avgdl <- mean(dl)       # avgdl: tamanho médio dos documentos do corpus

idf_bm25 <- log((N - documentos_com_termo + 0.5) / (documentos_com_termo + 0.5) + 1)

cat("\n============================================\n")
cat("       BM25 — TAMANHO E IDF PROBABILÍSTICO\n")
cat("============================================\n\n")

cat("Tamanho dos documentos (dl):\n")
print(dl)
cat("\nTamanho médio do corpus (avgdl):", round(avgdl, 2), "\n")

cat("\nComparando IDF clássico (seção 6) x IDF do BM25:\n")
comparacao_idf <- data.frame(
  idf_classico = round(idf[c("caf", "fort", "vicent")], 3),
  idf_bm25 = round(idf_bm25[c("caf", "fort", "vicent")], 3)
)
print(comparacao_idf)

### 14. Implementação do BM25

`bm25_doc(consulta, d)` é a tradução direta da fórmula do slide 20 para R, seguindo a
mesma estrutura da função apresentada na aula:

- tokeniza a consulta com `tokenizar_limpar` — a mesma "regra de ouro" da seção 11:
  a consulta precisa passar pelo mesmo pré-processamento (stemming incluso) que os
  documentos, senão "museu" não bate com o radical `mus` indexado na `tdm`;
- para cada termo, busca `f` (a frequência do termo no documento) diretamente na `tdm`
  já construída na seção 5 — não é preciso reconstruir nenhuma matriz;
- calcula `K`, a penalidade de tamanho, usando `dl` e `avgdl` da seção 13;
- soma `idf_bm25[t] * (f * (k1 + 1)) / (f + K)` ao acumulador `s`, exatamente como no
  slide 34-35.

`bm25_ranking(consulta)` aplica `bm25_doc` a cada documento do corpus e devolve os
escores em ordem decrescente — o ranking final para aquela consulta.

In [ ]:
k1 <- 1.2   # controla a saturação da frequência (padrão do slide 20)
b  <- 0.75  # controla o peso do tamanho do documento (padrão do slide 20)

bm25_doc <- function(consulta, d) {
  termos <- tokenizar_limpar(consulta)   # mesma regra de pré-processamento dos documentos
  s <- 0
  for (t in termos) {
    if (!(t %in% vocab)) next            # termo fora do vocabulário: contribui 0
    f <- tdm[t, d]                       # f: frequência do termo NESTE documento
    K <- k1 * (1 - b + b * dl[d] / avgdl)
    s <- s + idf_bm25[t] * (f * (k1 + 1)) / (f + K)
  }
  return(s)
}

bm25_ranking <- function(consulta) {
  scores <- sapply(colnames(tdm), function(d) bm25_doc(consulta, d))
  sort(scores, decreasing = TRUE)
}

cat("\n============================================\n")
cat("       TESTE: bm25_ranking('café museu')\n")
cat("============================================\n\n")
print(round(bm25_ranking("café museu"), 4))

### 15. Comparando BM25 com TF-IDF em 3 consultas

Para comparar com o TF-IDF em pé de igualdade, `tfidf_doc(consulta, d)` soma o TF-IDF
(seção 6) dos termos da consulta que existem naquele documento — o análogo mais direto
do somatório do BM25, mas sem saturação nem correção de tamanho. `tfidf_ranking` aplica
isso a todos os documentos, do mesmo jeito que `bm25_ranking` faz.

As três consultas de teste cobrem os três municípios do corpus, para que cada um tenha
chance de aparecer no topo de algum ranking.

In [ ]:
tfidf_doc <- function(consulta, d) {
  termos <- tokenizar_limpar(consulta)
  termos <- termos[termos %in% rownames(tfidf)]
  if (length(termos) == 0) return(0)
  sum(tfidf[termos, d])
}

tfidf_ranking <- function(consulta) {
  scores <- sapply(colnames(tfidf), function(d) tfidf_doc(consulta, d))
  sort(scores, decreasing = TRUE)
}

consultas_teste <- c("café museu", "forte artilharia", "porto são vicente")

cat("\n============================================\n")
cat("       BM25 vs TF-IDF — 3 CONSULTAS\n")
cat("============================================\n")

for (consulta in consultas_teste) {
  cat("\n--------------------------------------------\n")
  cat("Consulta:", consulta, "\n")
  cat("--------------------------------------------\n")
  cat("BM25:\n")
  print(round(bm25_ranking(consulta), 4))
  cat("\nTF-IDF:\n")
  print(round(tfidf_ranking(consulta), 4))
}

### 16. Variando k1 e b

Por fim, a tarefa pede para observar o efeito de `k1` e `b` na ordem dos resultados.
`testar_parametros` percorre combinações de valores, reatribui `k1` e `b` globalmente
(por isso o `<<-`), recalcula o ranking da mesma consulta para cada combinação e, ao
final, restaura os valores padrão (1,2 e 0,75) para não afetar o restante do notebook.

Vale reparar em dois casos-limite, como no slide 23: `k1 = 0` reduz o BM25 a uma busca
quase booleana (a frequência deixa de importar) e `b = 0` desliga a normalização por
tamanho — os documentos maiores (como São Vicente) deixam de ser penalizados.

In [ ]:
testar_parametros <- function(consulta, k1_vals, b_vals) {
  for (k1v in k1_vals) {
    for (bv in b_vals) {
      k1 <<- k1v
      b  <<- bv
      cat(sprintf("\nk1 = %.1f | b = %.2f\n", k1v, bv))
      print(round(bm25_ranking(consulta), 4))
    }
  }
  k1 <<- 1.2  # restaura o padrão
  b  <<- 0.75
}

cat("\n============================================\n")
cat("       EFEITO DE k1 E b NO RANKING\n")
cat("============================================\n")

testar_parametros("café museu", k1_vals = c(0, 1.2, 3), b_vals = c(0, 0.75, 1))